# Document Model Testing: BGE-M3 (retrieval) + Qwen3 (generation)

Flow: upload PDF -> strip references -> chunk -> embed (BGE-M3) -> retrieve -> generate RAG answer (Qwen3) -> generate direct answer (Qwen3) -> compare.

Make sure you're on a GPU runtime: `Runtime > Change runtime type > T4 GPU`.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
  # dependencies
!pip install -q transformers accelerate sentence-transformers pypdf FlagEmbedding bitsandbytes


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 382.2/382.2 kB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.7/247.7 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.8/947.8 kB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 78.4 MB/s eta 0:00:00


In [ ]:
# Config : all settings in one place, edit here instead of going through cells

EMBED_MODEL_NAME = "BAAI/bge-m3"
EMBED_MAX_LENGTH = 1024

GEN_MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"   # must include the -2507 suffix, plain "-Instruct" does not exist
GEN_MAX_NEW_TOKENS = 400
GEN_DTYPE = "float16"

CHUNK_SIZE = 800
CHUNK_OVERLAP = 100

TOP_K = 10
DIRECT_MAX_CHARS = 6000

# OUTPUT_DIR = "outputs"
OUTPUT_DIR = "/content/drive/MyDrive/doc-model-bench/outputs"


In [4]:
# Upload your PDF
from google.colab import files
uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]
print("Uploaded:", pdf_path)


Saving Hands-On-Large-Language-Models.pdf to Hands-On-Large-Language-Models.pdf
Uploaded: Hands-On-Large-Language-Models.pdf


In [5]:
# Extract text from the PDF
from pypdf import PdfReader

def extract_text_from_pdf(pdf_path: str) -> str:
    reader = PdfReader(pdf_path)
    full_text = []
    for page in reader.pages:
        full_text.append(page.extract_text() or "")
    combined = "\n".join(full_text)
    print(f"[extract] {len(combined)} characters from {len(reader.pages)} pages")
    return combined

full_text = extract_text_from_pdf(pdf_path)
print(full_text[:800])  # preview


[extract] 641582 characters from 598 pages

Praise for Hands-On Large Language Models
This is an exceptional guide to the world of language models and their
practical applications in industry. Its highly-visual coverage of
generative, representational, and retrieval applications of language
models empowers readers to quickly understand, use, and refine LLMs.
Highly recommended!
—Nils Reimers, Director of Machine Learning at Cohere |
creator of sentence-transformers
Jay and Maarten have continued their tradition of providing beautifully
illustrated and insightful descriptions of complex topics in their new
book. Bolstered with working code, timelines, and references to key
papers, their book is a valuable resource for anyone looking to
understand the main techniques behind how Large Language Models are
built.
—Andrew Ng, founder of 


In [6]:
import re

def remove_references_section(text: str) -> str:
    pattern = re.compile(
        r"\n\s*\d*\.?\s*(References|REFERENCES|Bibliography|BIBLIOGRAPHY)\s*\d*\.?\s*\n"
    )
    matches = list(pattern.finditer(text))
    if not matches:
        print("[references] No heading found — using full text as-is")
        return text
    cut_index = matches[-1].start()
    trimmed = text[:cut_index]
    print(f"[references] Removed {len(text) - len(trimmed)} characters after last References/Bibliography heading")
    return trimmed

full_text = remove_references_section(full_text)

[references] No heading found — using full text as-is


In [7]:
import re

for match in re.finditer(r"references", full_text, re.IGNORECASE):
    start = max(0, match.start() - 60)
    end = min(len(full_text), match.end() + 60)
    print(f"--- match at position {match.start()} ---")
    print(repr(full_text[start:end]))
    print()

--- match at position 620 ---
'their new\nbook. Bolstered with working code, timelines, and references to key\npapers, their book is a valuable resource for anyone'

--- match at position 50836 ---
'ps can be added to further align the model with\nthe user’s preferences, as we will explore in Chapter 12.\nLarge Language Model App'

--- match at position 93924 ---
'ocabulary size: 50,000\nSpecial tokens:\n<s>\n<pad>\n</s>\n<unk>\nReferences: Citations are wrapped within the two special tokens:\n[STAR'

--- match at position 286656 ---
', as it allows us to effectively communicate\nour needs and preferences to the model. By mastering prompt engineering\ntechniques, w'

--- match at position 482127 ---
'ric("f1")\n   f1 = load_f1.compute(predictions=predictions, \nreferences=labels)["f1"]\n   return {"f1": f1}\nWith compute_metrics we '

--- match at position 515212 ---
'esults = seqeval.compute(\n    predictions=true_predictions, references=true_labels\n)\n    return {"f1": results["over

In [8]:
# Chunk the text
def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        c = text[start:end].strip()
        if c:
            chunks.append(c)
        start += chunk_size - overlap
    return chunks

chunks = chunk_text(full_text)
print(f"[chunk] Created {len(chunks)} chunks")
print("Example chunk:\n", chunks[0][:300])


[chunk] Created 917 chunks
Example chunk:
 Praise for Hands-On Large Language Models
This is an exceptional guide to the world of language models and their
practical applications in industry. Its highly-visual coverage of
generative, representational, and retrieval applications of language
models empowers readers to quickly understand, use, 


In [9]:
# Embed chunks with BGE-M3
from FlagEmbedding import BGEM3FlagModel

embed_model = BGEM3FlagModel(EMBED_MODEL_NAME, use_fp16=True)
chunk_embeddings = embed_model.encode(chunks, batch_size=12, max_length=EMBED_MAX_LENGTH)['dense_vecs']
print("Embedded shape:", chunk_embeddings.shape)


config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Inference Embeddings: 100%|██████████| 77/77 [00:06<00:00, 12.06it/s]

Embedded shape: (917, 1024)


In [16]:
# Ask a question, retrieve relevant chunks
import numpy as np

question = "What is the main contribution of this paper?"
# question = "What is this paper about?"
# question = "who is the author of this paper?"

query_embedding = embed_model.encode([question], max_length=EMBED_MAX_LENGTH)['dense_vecs']

def cosine_sim(a, b):
    a = a / np.linalg.norm(a, axis=-1, keepdims=True)
    b = b / np.linalg.norm(b, axis=-1, keepdims=True)
    return a @ b.T

sims = cosine_sim(query_embedding, chunk_embeddings)[0]
top_indices = np.argsort(sims)[::-1][:TOP_K]
retrieved_chunks = [chunks[i] for i in top_indices]

print(f"Top {TOP_K} retrieved chunks:")
for rank, i in enumerate(top_indices):
    print(f"\n--- Rank {rank+1} (score {sims[i]:.3f}) ---")
    print(chunks[i][:300])


Top 10 retrieved chunks:

--- Rank 1 (score 0.533) ---
t! The first text generated in this book was a decent joke about
chickens.
Summary
In this first chapter of the book, we delved into the revolutionary impact
LLMs have had on the Language AI field. It has significantly changed our
approach to tasks such as translation, classification, summarization,

--- Rank 2 (score 0.510) ---
ocessing and
Context Size
recent improvements to, Recent Improvements to the Transformer
Architecture-Other Architectural Experiments and Improvements
Transformer blocks, Inside the Transformer Block-Self-attention:
Combining information
utility of, Large Language Model Applications: What Makes Them

--- Rank 3 (score 0.504) ---
d relevance scoring, Self-attention: Relevance
scoring-Self-attention: Combining information
Vision Transformer, Transformers for Vision-Transformers for Vision
transparency and accountability, Responsible LLM Development and Usage
tree-of-thought, Tree-of-Thought: Exploring Interme

In [12]:
# Load Qwen3 for generation
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL_NAME)
gen_model = AutoModelForCausalLM.from_pretrained(
    GEN_MODEL_NAME,
    torch_dtype=getattr(torch, GEN_DTYPE),
    device_map="auto"
)

def ask_qwen(prompt, max_new_tokens=GEN_MAX_NEW_TOKENS):
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(gen_model.device)
    output = gen_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.38k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

model.safetensors.index.json:   0%|          | 0.00/32.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

In [17]:
# RAG answer (retrieved chunks only)
context = "\n\n".join(retrieved_chunks)
rag_prompt = f'''Answer the question using ONLY the context below. Be concise.

Context:
{context}

Question: {question}
'''
rag_answer = ask_qwen(rag_prompt)
print("=== RAG Answer ===\n")
print(rag_answer)


=== RAG Answer ===

The context provided does not include a specific paper with a main contribution described. Therefore, based solely on the given information, the main contribution of any paper cannot be identified.


In [18]:
# Direct answer (raw truncated text, no retrieval)
direct_context = full_text[:DIRECT_MAX_CHARS]
direct_prompt = f'''Answer the question based on the document below. Be concise.

Document:
{direct_context}

Question: {question}
'''
direct_answer = ask_qwen(direct_prompt)
print("=== Direct Answer ===\n")
print(direct_answer)


=== Direct Answer ===

The document is not a paper but a book review and praise for *Hands-On Large Language Models* by Jay Alammar and Maarten Grootendorst. It does not present a research paper with a main contribution. Therefore, there is no "main contribution" of a paper in this document.


In [19]:
# Side-by-side comparison
import os
from datetime import datetime

print("QUESTION:", question)
print("\n" + "="*60)
print("RAG ANSWER:")
print("="*60)
print(rag_answer)
print("\n" + "="*60)
print("DIRECT ANSWER:")
print("="*60)
print(direct_answer)

os.makedirs(OUTPUT_DIR, exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
base_name = os.path.splitext(os.path.basename(pdf_path))[0]
md_path = os.path.join(OUTPUT_DIR, f"{base_name}_{timestamp}.md")

with open(md_path, "w") as f:
    f.write(f"# Test log: {base_name}\n\n")
    f.write(f"**Date:** {datetime.now().strftime('%Y-%m-%d %H:%M')}\n\n")
    f.write(f"**Question:** {question}\n\n")
    f.write("## RAG answer\n\n" + rag_answer + "\n\n")
    f.write("## Direct answer\n\n" + direct_answer + "\n\n")
    f.write("## Retrieved chunks\n\n")
    for i, (chunk, score) in enumerate(zip(retrieved_chunks, [sims[i] for i in top_indices])):
        f.write(f"**Chunk {i+1}** (similarity: {score:.3f})\n\n> {chunk[:300]}...\n\n")

print(f"\nSaved to {md_path}")


QUESTION: What is the main contribution of this paper?

RAG ANSWER:
The context provided does not include a specific paper with a main contribution described. Therefore, based solely on the given information, the main contribution of any paper cannot be identified.

DIRECT ANSWER:
The document is not a paper but a book review and praise for *Hands-On Large Language Models* by Jay Alammar and Maarten Grootendorst. It does not present a research paper with a main contribution. Therefore, there is no "main contribution" of a paper in this document.

Saved to /content/drive/MyDrive/doc-model-bench/outputs/Hands-On-Large-Language-Models_20260814_051144.md
